# 2Day Start Cross-Dataset Validation (A->B / B->A)

This notebook is adapted from run_train_pipe_1day_cross_dataset_validation.ipynb for 2Day Start evaluation.

Goal:
- Pick 2 dataset files from config
- Use day=1 rows for 2Day Start setting
- Train on dataset A and validate on dataset B
- Train on dataset B and validate on dataset A
- Compare weighted F1 and per-class F1
- Evaluate SEER/WEREWOLF/VILLAGER/POSSESSED constrained metrics with role-assignment logic

In [6]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(pd.DataFrame([
    {"key": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
    {"key": "CONFIG_PATH", "value": str(CONFIG_PATH)},
    {"key": "python", "value": sys.version.split()[0]},
]))

            key                                              value
0  PROJECT_ROOT  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
1   CONFIG_PATH  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
2        python                                             3.13.1


In [21]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# プロジェクトルートとsrcディレクトリの絶対パスをsys.pathに追加
PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.resolve()))
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))

from src.pipelines.training_pipeline import load_training_config, get_data_paths, get_models_dir
from src.Rolepredicter.role_assignment import assign_roles_for_non_seer, assign_roles_for_seer_by_divination
print("Module import complete.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Module import complete.


In [13]:
RUN_OPTIONS = {
    "run_training": True,
    "top_k_features": 10,
    "show_only_required_artifacts": True,
    "day2_flag": True,  # 2日目脱落者制約を有効
    "tuning_cv_splits": 3,
    "train_index": 0,
    "test_index": 1,
    "n_trials": 30,
    "tuning_cv_splits": 3,
    "optuna_seed": 42
}


config = load_training_config(CONFIG_PATH)


data_paths = get_data_paths(config, data_type="day2")

if len(data_paths) < 2:
    raise ValueError("Need at least 2 data files in training_config.json -> data_paths.")

train_idx = int(RUN_OPTIONS["train_index"])
test_idx = int(RUN_OPTIONS["test_index"])
if train_idx == test_idx:
    raise ValueError("train_index and test_index must be different.")

print("="*70)
print("2DAY START パイプライン設定")
print("="*70)
print("\nRun options:")
print(json.dumps(RUN_OPTIONS, ensure_ascii=False, indent=2))
print("\nTraining config (effective):")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("\n2Day Start override:")
print(json.dumps(override, ensure_ascii=False, indent=2))
print(f"\nEffective n_trials: {config['n_trials']}")
print("\nFeature policy:")
if bool(config.get("lang_feature", False)):
    print("- lang_feature=True ですが、話者推定モデル/話者推定確率特徴は無効です。")
else:
    print("- 話者推定モデル/話者推定確率特徴は使用しません。")

selected_paths = [data_paths[train_idx], data_paths[test_idx]]

display(pd.DataFrame([
    {"name": "train_candidate", "index": train_idx, "path": selected_paths[0]},
    {"name": "test_candidate", "index": test_idx, "path": selected_paths[1]},
    {"name": "day2_flag", "index": "-", "path": RUN_OPTIONS["day2_flag"]},
    {"name": "n_trials", "index": "-", "path": RUN_OPTIONS["n_trials"]},
    {"name": "tuning_cv_splits", "index": "-", "path": RUN_OPTIONS["tuning_cv_splits"]},
]))

✓ Data files selected:
  - all_feature_table_2025sp17_day2_with_talks.csv
  - all_feature_table_2025summer_day2_with_talks.csv
2DAY START パイプライン設定

Run options:
{
  "run_training": true,
  "top_k_features": 10,
  "show_only_required_artifacts": true,
  "day2_flag": true,
  "tuning_cv_splits": 3,
  "train_index": 0,
  "test_index": 1,
  "n_trials": 30,
  "optuna_seed": 42
}

Training config (effective):
{
  "data_paths": [
    "data/2025spring/all_feature_table_2025sp17_with_talks.csv",
    "data/2025summer/all_feature_table_2025summer_with_talks2.csv"
  ],
  "data_paths_day2": [
    "data/2025spring/all_feature_table_2025sp17_day2_with_talks.csv",
    "data/2025summer/all_feature_table_2025summer_day2_with_talks.csv"
  ],
  "day_filter": 1,
  "lang_feature": false,
  "group_column": "source_file",
  "cv_folds": 5,
  "test_size": 0.2,
  "n_trials": 20,
  "leakage_drop_columns": [
    "True_Div_recepient_id_1",
    "True_Div_result_1",
    "True_Div_recepient_id_2",
    "True_Div_result_

,name,index,path
0,train_candidate,0,C:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
1,test_candidate,1,C:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
2,day2_flag,-,True
3,n_trials,-,30
4,tuning_cv_splits,-,3


In [17]:
base_drop_cols = [
    "id", "role", "source_file", "dataset_tag", "day", "role_encoded",
    "Est_id_Fact_role", "Est_id_Est_roles", "character_name",
    "agent_name", "combined_text", "seer_co_order"
]
meta_raw_cols = [
    "True_Div_result_1", "True_Div_recepient_id_1",
    "True_Div_result_2", "True_Div_recepient_id_2",
    "exec_id", "attack_id"
]
drop_cols = list(set(base_drop_cols + meta_raw_cols))

def load_df(path):
    path_obj = Path(path)
    df_raw = pd.read_csv(path)

    df_day= df_raw

    before_rows = len(df_day)
    if "attack_id" in df_day.columns:
        df_day = df_day.dropna(subset=["attack_id"]).copy()
    after_attack_filter_rows = len(df_day)

    removed_sources_for_seer = 0
    if "True_Div_result_1" in df_day.columns and "source_file" in df_day.columns:
        del_paths = df_day[(df_day["role"] == "SEER") & (df_day["True_Div_result_1"].isna())]["source_file"].unique()
        removed_sources_for_seer = len(del_paths)
        if removed_sources_for_seer > 0:
            df_day = df_day[~df_day["source_file"].isin(del_paths)].copy()

    if df_day.empty:
        raise ValueError(f"No usable rows remain after 2Day filters in {path_obj.name}")

    df_day["dataset_tag"] = path_obj.stem
    print(
        f"loaded {path_obj.name}: raw={len(df_raw)} day={before_rows} ",
        f"after_attack_filter={after_attack_filter_rows} final={len(df_day)} ",
        f"removed_seer_sources={removed_sources_for_seer}",
    )
    return df_day

df_a = load_df(selected_paths[0])
df_b = load_df(selected_paths[1])

summary_df = pd.DataFrame([
    {"dataset": "A", "tag": df_a["dataset_tag"].iloc[0], "rows": len(df_a), "path": selected_paths[0]},
    {"dataset": "B", "tag": df_b["dataset_tag"].iloc[0], "rows": len(df_b), "path": selected_paths[1]},
])
display(summary_df)

loaded all_feature_table_2025sp17_day2_with_talks.csv: raw=530 day=530  after_attack_filter=335 final=335  removed_seer_sources=0
loaded all_feature_table_2025summer_day2_with_talks.csv: raw=595 day=595  after_attack_filter=410 final=375  removed_seer_sources=7


,dataset,tag,rows,path
0,A,all_feature_table_2025sp17_day2_with_talks,335,C:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
1,B,all_feature_table_2025summer_day2_with_talks,375,C:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...


In [18]:
# Build shared label space and shared feature columns
all_roles = sorted(pd.concat([df_a["role"], df_b["role"]]).astype(str).dropna().unique().tolist())
label_encoder = LabelEncoder()
label_encoder.fit(all_roles)

feature_cols_a = [c for c in df_a.columns if c not in drop_cols]
feature_cols_b = [c for c in df_b.columns if c not in drop_cols]
final_features = sorted(set(feature_cols_a).union(set(feature_cols_b)))

def make_bundle(df):
    x_df = df.drop(columns=drop_cols, errors="ignore").reindex(columns=final_features, fill_value=0)
    x = x_df.apply(pd.to_numeric, errors="coerce").fillna(0).values
    y = label_encoder.transform(df["role"].astype(str).values)

    source_files = (
        df["source_file"].astype(str).values
        if "source_file" in df.columns
        else np.array(["unknown_source"] * len(df), dtype=object)
    )

    meta = {
        "div_result1": df["True_Div_result_1"].values if "True_Div_result_1" in df.columns else np.full(len(df), np.nan),
        "div_id1": df["True_Div_recepient_id_1"].values if "True_Div_recepient_id_1" in df.columns else np.full(len(df), np.nan),
        "div_result2": df["True_Div_result_2"].values if "True_Div_result_2" in df.columns else np.full(len(df), np.nan),
        "div_id2": df["True_Div_recepient_id_2"].values if "True_Div_recepient_id_2" in df.columns else np.full(len(df), np.nan),
        "exec_id": df["exec_id"].values if "exec_id" in df.columns else np.full(len(df), np.nan),
        "attack_id": df["attack_id"].values if "attack_id" in df.columns else np.full(len(df), np.nan),
    }

    return {
        "X": x,
        "y": y,
        "meta": meta,
        "source_files": source_files,
    }

bundle_a = make_bundle(df_a)
bundle_b = make_bundle(df_b)

print(f"features: {len(final_features)}")
print(f"labels: {list(label_encoder.classes_)}")
print(f"A samples: {len(bundle_a['y'])}")
print(f"B samples: {len(bundle_b['y'])}")

features: 438
labels: [np.str_('POSSESSED'), np.str_('SEER'), np.str_('VILLAGER'), np.str_('WEREWOLF')]
A samples: 335
B samples: 375


In [19]:
VIEWS = ["SEER", "WEREWOLF", "VILLAGER", "POSSESSED"]
ROLE_COUNTS = {"POSSESSED": 1, "SEER": 1, "VILLAGER": 2, "WEREWOLF": 1}

def tune_xgb_params(train_x, train_y, n_trials=30, cv_splits=3, seed=42):
    n_splits = int(max(2, min(cv_splits, np.min(np.bincount(train_y)) if len(np.unique(train_y)) > 1 else 2)))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    def objective(trial):
        params = {
            "objective": "multi:softprob",
            "num_class": len(label_encoder.classes_),
            "eval_metric": "mlogloss",
            "tree_method": "hist",
            "random_state": seed,
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 150, 700),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }

        fold_scores = []
        for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(train_x, train_y), start=1):
            x_tr, x_va = train_x[tr_idx], train_x[va_idx]
            y_tr, y_va = train_y[tr_idx], train_y[va_idx]

            classes = np.unique(y_tr)
            class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
            weight_map = dict(zip(classes, class_weights))
            sample_weight = np.array([weight_map[label] for label in y_tr], dtype=float)

            model = xgb.XGBClassifier(**params)
            model.fit(x_tr, y_tr, sample_weight=sample_weight, verbose=False)
            pred = model.predict(x_va)
            score = f1_score(y_va, pred, average="macro", zero_division=0)
            fold_scores.append(score)

            trial.report(float(np.mean(fold_scores)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=int(max(1, n_trials)), show_progress_bar=False)

    best_params = study.best_params.copy()
    best_params.update({
        "objective": "multi:softprob",
        "num_class": len(label_encoder.classes_),
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": seed,
    })

    return best_params, study

def build_model(train_x, train_y):
    best_params, study = tune_xgb_params(
        train_x,
        train_y,
        n_trials=RUN_OPTIONS["n_trials"],
        cv_splits=RUN_OPTIONS["tuning_cv_splits"],
        seed=RUN_OPTIONS["optuna_seed"],
    )

    classes = np.unique(train_y)
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_y)
    weight_map = dict(zip(classes, class_weights))
    sample_weight = np.array([weight_map[label] for label in train_y], dtype=float)

    model = xgb.XGBClassifier(**best_params)
    model.fit(train_x, train_y, sample_weight=sample_weight, verbose=False)
    return model, best_params, float(study.best_value), int(study.best_trial.number)

def evaluate_view_with_constraints(view_name, preds_proba, y_true, meta, source_files):
    all_p, all_t = [], []
    eval_games = np.unique(source_files)

    for game_id in eval_games:
        game_mask = source_files == game_id
        game_indices = np.where(game_mask)[0]

        if len(game_indices) < 2:
            continue

        preds_game = preds_proba[game_indices]
        y_game = y_true[game_indices]
        meta_game = {k: v[game_indices] for k, v in meta.items()}

        if view_name == "SEER":
            res = assign_roles_for_seer_by_divination(
                preds_game, y_game, ROLE_COUNTS, label_encoder.classes_, label_encoder,
                meta_game["div_result1"], meta_game["div_id1"],
                meta_game["div_result2"], meta_game["div_id2"],
                meta_game["exec_id"], meta_game["attack_id"],
                RUN_OPTIONS["day2_flag"],
            )
            for key in ["black", "white"]:
                if res[key][0].size > 0:
                    all_p.extend(res[key][0])
                    all_t.extend(res[key][1])
        else:
            p, t = assign_roles_for_non_seer(
                preds_game, y_game, ROLE_COUNTS, label_encoder.classes_, label_encoder,
                view_name, meta_game["exec_id"], meta_game["attack_id"],
                RUN_OPTIONS["day2_flag"],
            )
            if p.size > 0:
                all_p.extend(p)
                all_t.extend(t)

    if len(all_t) == 0:
        return 0.0, 0

    target_label = "POSSESSED" if view_name == "WEREWOLF" else "WEREWOLF"
    target_id = label_encoder.transform([target_label])[0]
    score = f1_score(all_t, all_p, labels=[target_id], average="macro", zero_division=0)
    return float(score), len(all_t)

def train_and_validate(train_bundle, test_bundle, train_name, test_name):
    model, best_params, best_cv_score, best_trial = build_model(train_bundle["X"], train_bundle["y"])

    pred = model.predict(test_bundle["X"])
    pred_proba = model.predict_proba(test_bundle["X"])

    weighted_f1 = f1_score(test_bundle["y"], pred, average="weighted", zero_division=0)
    macro_f1 = f1_score(test_bundle["y"], pred, average="macro", zero_division=0)

    report_dict = classification_report(
        test_bundle["y"],
        pred,
        target_names=label_encoder.classes_,
        zero_division=0,
        output_dict=True,
    )

    per_class_rows = []
    for cls_name in label_encoder.classes_:
        cls_report = report_dict.get(cls_name, {})
        per_class_rows.append({
            "train_on": train_name,
            "validate_on": test_name,
            "role": cls_name,
            "precision": cls_report.get("precision", 0.0),
            "recall": cls_report.get("recall", 0.0),
            "f1": cls_report.get("f1-score", 0.0),
            "support": cls_report.get("support", 0),
        })

    constrained_rows = []
    for view in VIEWS:
        constrained_f1, n_assigned = evaluate_view_with_constraints(
            view, pred_proba, test_bundle["y"], test_bundle["meta"], test_bundle["source_files"]
        )
        constrained_rows.append({
            "train_on": train_name,
            "validate_on": test_name,
            "view": view,
            "constrained_f1": constrained_f1,
            "assigned_samples": int(n_assigned),
        })

    summary = {
        "train_on": train_name,
        "validate_on": test_name,
        "weighted_f1": float(weighted_f1),
        "macro_f1": float(macro_f1),
        "optuna_best_cv_macro_f1": float(best_cv_score),
        "optuna_best_trial": int(best_trial),
        "optuna_best_params": json.dumps(best_params, ensure_ascii=False),
        "n_train": int(len(train_bundle["y"])),
        "n_test": int(len(test_bundle["y"])),
    }

    return model, summary, pd.DataFrame(per_class_rows), pd.DataFrame(constrained_rows), best_params

In [22]:
tag_a = str(df_a["dataset_tag"].iloc[0])
tag_b = str(df_b["dataset_tag"].iloc[0])

model_a_to_b, summary_a_to_b, class_a_to_b, constrained_a_to_b, best_params_a_to_b = train_and_validate(
    bundle_a, bundle_b, train_name=tag_a, test_name=tag_b
)
model_b_to_a, summary_b_to_a, class_b_to_a, constrained_b_to_a, best_params_b_to_a = train_and_validate(
    bundle_b, bundle_a, train_name=tag_b, test_name=tag_a
)

summary_cross_df = pd.DataFrame([summary_a_to_b, summary_b_to_a])
per_class_cross_df = pd.concat([class_a_to_b, class_b_to_a], ignore_index=True)
constrained_cross_df = pd.concat([constrained_a_to_b, constrained_b_to_a], ignore_index=True)

best_params_df = pd.DataFrame([
    {"train_on": tag_a, "validate_on": tag_b, **best_params_a_to_b},
    {"train_on": tag_b, "validate_on": tag_a, **best_params_b_to_a},
])

print("Cross validation finished.")
print("Standard metrics:")
display(summary_cross_df)
print("Optuna best params:")
display(best_params_df)
print("Per-class metrics:")
display(per_class_cross_df)
print("Constraint-based metrics (view-specific):")
display(constrained_cross_df)

Cross validation finished.
Standard metrics:


,train_on,validate_on,weighted_f1,macro_f1,optuna_best_cv_macro_f1,optuna_best_trial,optuna_best_params,n_train,n_test
0,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,0.410007,0.364918,0.472915,14,"{""max_depth"": 3, ""learning_rate"": 0.1026773050...",335,375
1,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,0.471352,0.427153,0.494312,25,"{""max_depth"": 6, ""learning_rate"": 0.0337236775...",375,335


Optuna best params:


,train_on,validate_on,max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,objective,num_class,eval_metric,tree_method,random_state
0,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,3,0.102677,605,20,0.856555,0.768215,6.585970e-06,4.874210e-07,0.022098,multi:softprob,4,mlogloss,hist,42
1,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,6,0.033724,392,1,0.876721,0.620071,7.585108e-08,4.972741e-02,0.001057,multi:softprob,4,mlogloss,hist,42


Per-class metrics:


,train_on,validate_on,role,precision,recall,f1,support
0,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,POSSESSED,0.230769,0.240000,0.235294,75.0
1,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,SEER,0.428571,0.200000,0.272727,75.0
2,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,VILLAGER,0.538462,0.653333,0.590361,150.0
3,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,WEREWOLF,0.350000,0.373333,0.361290,75.0
4,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,POSSESSED,0.342105,0.194030,0.247619,67.0
5,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,SEER,0.548387,0.507463,0.527132,67.0
6,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,VILLAGER,0.552632,0.783582,0.648148,134.0
7,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,WEREWOLF,0.355556,0.238806,0.285714,67.0


Constraint-based metrics (view-specific):


,train_on,validate_on,view,constrained_f1,assigned_samples
0,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,SEER,0.925926,108
1,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,WEREWOLF,0.320000,300
2,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,VILLAGER,0.517241,348
3,all_feature_table_2025sp17_day2_with_talks,all_feature_table_2025summer_day2_with_talks,POSSESSED,0.421053,152
4,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,SEER,1.000000,84
5,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,WEREWOLF,0.402985,268
6,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,VILLAGER,0.600000,340
7,all_feature_table_2025summer_day2_with_talks,all_feature_table_2025sp17_day2_with_talks,POSSESSED,0.562500,128


In [7]:
# Optional: save outputs
base_output_dir = PROJECT_ROOT / "notebooks" / "outputs" / "2day_start_cross_dataset_validation"
base_output_dir.mkdir(parents=True, exist_ok=True)

summary_path = base_output_dir / "cross_dataset_summary_2day_start.csv"
per_class_path = base_output_dir / "cross_dataset_per_class_2day_start.csv"
constrained_path = base_output_dir / "cross_dataset_constrained_by_view_2day_start.csv"
models_path = base_output_dir / "cross_dataset_models_2day_start.joblib"
meta_path = base_output_dir / "cross_dataset_meta_2day_start.json"

summary_cross_df.to_csv(summary_path, index=False)
per_class_cross_df.to_csv(per_class_path, index=False)
constrained_cross_df.to_csv(constrained_path, index=False)

import joblib
joblib.dump({
    "model_a_to_b": model_a_to_b,
    "model_b_to_a": model_b_to_a,
    "label_classes": list(label_encoder.classes_),
    "features": final_features,
}, models_path)

meta = {
    "mode": "2day_start",
    "selected_paths": selected_paths,
    "day_filter": RUN_OPTIONS["day_filter"],
    "day2_flag": RUN_OPTIONS["day2_flag"],
    "dataset_tags": [tag_a, tag_b],
    "n_features": len(final_features),
    "views": VIEWS,
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Saved: {summary_path}")
print(f"Saved: {per_class_path}")
print(f"Saved: {constrained_path}")
print(f"Saved: {models_path}")
print(f"Saved: {meta_path}")

Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start_cross_dataset_validation\cross_dataset_summary_2day_start.csv
Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start_cross_dataset_validation\cross_dataset_per_class_2day_start.csv
Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start_cross_dataset_validation\cross_dataset_constrained_by_view_2day_start.csv
Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start_cross_dataset_validation\cross_dataset_models_2day_start.joblib
Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start_cross_dataset_validation\cross_dataset_meta_2day_start.json
